In [1]:
import sys
import os
sys.path.append(os.path.abspath("../Code"))
import preproc
from transformers import pipeline
from sklearn.metrics import f1_score
import pandas as pd

In [ ]:
df = pd.read_csv("../Data/raw/tcga_simple_dev.csv")

print("Preprocesando...")
df["text"] = preproc.preprocesamiento(df["text"],3)

ruta_modelo_local = "../Models/longformer-prep_3_aug_len_1536"
nlp_pipeline = pipeline("text-classification", model=ruta_modelo_local, tokenizer=ruta_modelo_local)


textos = df['text'].tolist()
print("Infiriendo...")
resultados_pipeline = nlp_pipeline(textos, batch_size=8)

predicciones = [res['label'] for res in resultados_pipeline]

label_map = {'LABEL_0': 'T1', 'LABEL_1': 'T2', 'LABEL_2': 'T3', 'LABEL_3': 'T4'}
predicciones = [label_map[pred] for pred in predicciones]

etiquetas_reales = df['t'].tolist()

f1_macro = f1_score(etiquetas_reales, predicciones, average='macro')
print(f"F1-Score (macro): {f1_macro:.4f}")

Preprocesando...


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: ../Models/longformer-prep_3_aug_len_1536
Key                  | Status     |  | 
---------------------+------------+--+-
loss_function.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Infiriendo...
F1-Score (macro): 0.8313


In [6]:
df2 = pd.read_csv("../Data/raw/tcga_simple_test_empty.csv") 

print("Preprocesando df2...")
df2["text_proc"] = preproc.preprocesamiento(df2["text"], 3)

# 2. Extraer los textos y pasarlos por el pipeline que ya habías cargado
textos_df2 = df2['text_proc'].tolist()

print("Infiriendo sobre df2...")
# Usamos el mismo nlp_pipeline y batch_size para acelerar el proceso
resultados_pipeline_df2 = nlp_pipeline(textos_df2, batch_size=8)

# 3. Extraer las etiquetas crudas ('LABEL_X')
predicciones_df2 = [res['label'] for res in resultados_pipeline_df2]

# 4. Mapear las etiquetas a los nombres reales ('T1', 'T2', etc.)
label_map = {'LABEL_0': 'T1', 'LABEL_1': 'T2', 'LABEL_2': 'T3', 'LABEL_3': 'T4'}
predicciones_mapeadas = [label_map[pred] for pred in predicciones_df2]

# 5. Añadir la columna "t" con los resultados finales al df2
df2['t'] = predicciones_mapeadas
df2.drop(columns=['text_proc'], inplace=True)

print("¡Predicciones añadidas con éxito!")
df2.to_csv("../Results/Outputs/tcga_simple_test_full.csv", index=False)

Preprocesando df2...
Infiriendo sobre df2...
¡Predicciones añadidas con éxito!
